In [1]:
%reload_ext autoreload

%autoreload 2

import os.path as osp
from omegaconf.omegaconf import OmegaConf
import logging 
logging.getLogger().setLevel(logging.ERROR)

import sys
sys.path.append('externals/frankmocap/')
sys.path.append('externals/frankmocap/detectors/body_pose_estimator/')

from config.defaults import get_cfg_defaults

from nnutils.hand_utils import ManopthWrapper
from nnutils.handmocap import get_handmocap_predictor, process_mocap_predictions, get_handmocap_detector
from nnutils.hoiapi import vis_hand_object, Predictor
from nnutils import model_utils
from nnutils import image_utils

from externals.frankmocap.mocap_utils.demo_utils import extract_mesh_from_output

import numpy as np
from glob import glob
from PIL import Image
from matplotlib import pyplot as plt
%matplotlib inline

from renderer.screen_free_visualizer import Visualizer

import ipywidgets as widgets
import torch
assert torch.cuda.is_available()

In [2]:
!jupyter nbextension enable --py jupyter_bbox_widget

Enabling notebook extension jupyter_bbox_widget/extension...
      - Validating: OK


In [ ]:
# load image
filename = 'demo/milk.jpg'
image_pil = Image.open(filename).convert("RGB")

max_dim = 1920
# if either dimension is greater than max_dim, resize the image
if image_pil.size[0] > max_dim or image_pil.size[1] > max_dim:
    ratio = float(max_dim) / max(image_pil.size)
    image_pil = image_pil.resize((int(image_pil.size[0] * ratio), int(image_pil.size[1] * ratio)), Image.BILINEAR)

image = np.array(image_pil)
plt.close()
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
# Detect hands using FrankMocap
view_type = 'ego_centric' # 'third_view' or 'ego_centric'

object_mask = np.ones_like(image[..., 0])
visualizer = Visualizer('pytorch3d')
bbox_detector = get_handmocap_detector(view_type)

# Process Human Estimations.
detect_output = bbox_detector.detect_hand_bbox(image[..., ::-1].copy())
body_pose_list, body_bbox_list, hand_bbox_list, raw_hand_bboxes = detect_output

# visualize
res_img = visualizer.visualize(
    image, 
    hand_bbox_list = hand_bbox_list)

plt.close()
plt.imshow(res_img)
plt.axis('off')
plt.show()

In [ ]:

hand_predictor = get_handmocap_predictor()
mocap_predictions = hand_predictor.regress(
    image[..., ::-1], hand_bbox_list
)

hand_wrapper = ManopthWrapper().to('cpu')

pred_mesh_list = extract_mesh_from_output(mocap_predictions)

# visualize
res_img = visualizer.visualize(
    image, 
    pred_mesh_list = pred_mesh_list, 
    hand_bbox_list = hand_bbox_list)


plt_img = Image.fromarray(res_img.astype(np.uint8))
plt.close()
plt.imshow(plt_img)
plt.axis('off')
plt.show()

In [6]:

def get_hoi_predictor(experiment_directory):
    cfg_def = get_cfg_defaults()
    cfg_def = OmegaConf.create(cfg_def.dump())
    cfg = OmegaConf.load(osp.join(experiment_directory, 'hparams.yaml'))
    cfg = OmegaConf.merge(cfg_def, cfg)
    cfg.MODEL.BATCH_SIZE = 1
    model = model_utils.load_model(cfg, experiment_directory, 'last')

    predictor = Predictor(model)
    return predictor

In [7]:
data = process_mocap_predictions(
    mocap_predictions, image, hand_wrapper, mask=object_mask
)

exp_dir = './weights/horse' # change this path as required
new_predict = get_hoi_predictor(experiment_directory=exp_dir)

new_output = new_predict.forward_to_mesh(data)

In [ ]:
img_dir = osp.dirname(filename)
obj_name = osp.basename(filename).split('.')[0]
out_dir = osp.join(img_dir, obj_name+'_out')
vis_hand_object(new_output, data, image, out_dir + '/test')
Image.open(osp.join(out_dir, 'test_cHoi.png'))

In [ ]:
image_utils.display_gif(osp.join(out_dir, 'test_cHoi.gif'))